# Assignment 2B — Retrieval-Augmented Generation (RAG) Pipeline
**Group No. 7**  
**Course:** LLM4GenAI  
**Domain:** Financial Annual Reports (Apple, Amazon, NVIDIA, Tesla, Berkshire Hathaway)

---

## Pipeline Overview
```
Domain .txt Corpus
       ↓
  Part A: Chunking (Fixed-Size / Sliding Window / Semantic)
       ↓
  Part B: Retrieval (Dense FAISS / Sparse BM25 / Hybrid RRF)
       ↓
  Part C1: Cross-Encoder Reranking
  Part C2: Tabular RAG (PDF tables → serialised rows → indexed)
```

---
## 📦 Step 1.1 — Install Dependencies

Install all required libraries. Run this cell first.
- `sentence-transformers` — for dense embeddings
- `faiss-cpu` — vector index for dense retrieval
- `rank_bm25` — BM25 sparse retrieval
- `pdfplumber` — extract tables from PDFs
- `transformers` — cross-encoder reranking model
- `nltk` — sentence tokenisation for semantic chunking

In [1]:
# Pin three packages to specific versions to avoid compatibility breaks.
# sentence-transformers==2.7.0 : embedding model library; 3.x changed the API
# transformers==4.40.2         : required by sentence-transformers internally
# protobuf==3.20.3             : needed by transformers; newer versions break deserialization
# -q flag : quiet mode — suppresses verbose install output so the cell stays readable
pip install -q "sentence-transformers==2.7.0" "transformers==4.40.2" "protobuf==3.20.3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.21.0 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
googleapis-common-protos 1.75.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-spanner 3.68.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which 

In [2]:
import sys  # sys.executable gives us the exact Python binary path for this environment

# Install all required libraries for the RAG pipeline.
# Using sys.executable ensures pip installs into the same Python environment
# that this notebook is running in — avoids "module not found" after install.
# --quiet suppresses the verbose installation log so the cell output stays clean.
!{sys.executable} -m pip install sentence-transformers faiss-cpu rank_bm25 pdfplumber \
    pandas numpy transformers nltk tqdm --quiet
# sentence-transformers : load and run the all-MiniLM-L6-v2 embedding model
# faiss-cpu             : Facebook AI Similarity Search — vector index for dense retrieval
# rank_bm25             : BM25Okapi sparse retrieval index
# pdfplumber            : extract tables from PDF pages (used in Group 6)
# pandas                : DataFrames for displaying comparison tables
# numpy                 : fast numerical arrays; used for embedding matrices and normalisation
# transformers          : HuggingFace model hub; required by sentence-transformers internally
# nltk                  : Natural Language Toolkit; provides sent_tokenize for semantic chunker
# tqdm                  : progress bars for long embedding loops

print("✅ All dependencies installed successfully.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ All dependencies installed successfully.


---
## 📂 Step 1.2 — Load Corpus

Unzip the domain corpus from Assignment 1A. It contains 5 cleaned financial annual report
text files: Apple, Amazon, NVIDIA, Tesla, and Berkshire Hathaway.
We load each file separately so we can track which company each chunk comes from.

In [3]:
import zipfile   # zipfile: standard library module for reading/extracting .zip archives
import os        # os: file and directory operations (path checks, makedirs, listdir)
import pandas as pd  # pandas: for building the corpus stats summary table

# Path to the corpus zip file uploaded to Colab's root directory (/content/)
# This zip contains 5 .txt files — one per company (apple, amazon, nvidia, tesla, berkshire)
CORPUS_ZIP = "domain_corpus (2).zip"  # the filename as uploaded to Colab

# Destination folder where we will extract the .txt files
CORPUS_DIR = "corpus"

# Create the output folder if it doesn't already exist
# exist_ok=True: no error if the folder is already there (safe to re-run)
os.makedirs(CORPUS_DIR, exist_ok=True)

# Extract the zip archive into CORPUS_DIR
# 'r' mode: open zip for reading (not writing)
# extractall(): unzip every file inside into the target folder
with zipfile.ZipFile(CORPUS_ZIP, 'r') as z:
    z.extractall(CORPUS_DIR)

# ── Load each extracted .txt file into memory ─────────────────────────────────
corpus_files = {}  # dict: company_name → full text string (e.g. 'nvidia' → '...92k words...')
stats = []         # list of dicts: one entry per company with word/char counts for the summary table

# os.listdir() returns all filenames in CORPUS_DIR; sorted() gives alphabetical order
for fname in sorted(os.listdir(CORPUS_DIR)):
    if fname.endswith('.txt'):  # skip any non-text files (e.g. __MACOSX metadata from zip)
        fpath = os.path.join(CORPUS_DIR, fname)  # full path: e.g. "corpus/nvidia.txt"

        # Open with utf-8 encoding; errors='ignore' skips any non-UTF8 bytes silently
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()  # read entire file as one string

        # Derive company name by stripping the .txt extension
        company = fname.replace('.txt', '')  # e.g. 'nvidia.txt' → 'nvidia'

        # Count words by splitting on whitespace — a quick proxy for document length
        word_count = len(text.split())

        # Store the full text for later use in chunking
        corpus_files[company] = text

        # Record stats for the summary table printed below
        stats.append({'Company': company, 'File': fname, 'Words': word_count, 'Chars': len(text)})

        print(f"  Loaded {company:<15} → {word_count:>7,} words")

# ── Build full corpus string (for any whole-corpus operations) ────────────────
# Join all 5 texts with double newline between companies
# This is used for word count totals; chunking is done per-company separately
full_corpus = "\n\n".join(corpus_files.values())
total_words = sum(s['Words'] for s in stats)  # sum of word counts across all companies

# Print overall summary
print(f"\n{'='*45}")
print(f"  Total documents : {len(corpus_files)}")  # should be 5
print(f"  Total words     : {total_words:,}")       # expected ~222,925
print(f"  Total chars     : {len(full_corpus):,}")  # character count of combined corpus
print(f"{'='*45}")

# Display as a formatted table for easy reading in the notebook
df_corpus_stats = pd.DataFrame(stats)  # convert list of dicts → DataFrame (rows=companies)
display(df_corpus_stats)               # renders as an HTML table in Jupyter/Colab


  Loaded amazon          →  42,077 words
  Loaded apple           →  41,760 words
  Loaded berkshire       →  40,812 words
  Loaded nvidia          →  92,722 words
  Loaded tesla           →   5,554 words

  Total documents : 5
  Total words     : 222,925
  Total chars     : 1,733,315


,Company,File,Words,Chars
0,amazon,amazon.txt,42077,313389
1,apple,apple.txt,41760,269926
2,berkshire,berkshire.txt,40812,510403
3,nvidia,nvidia.txt,92722,605788
4,tesla,tesla.txt,5554,33801


---
## 🌐 Step 1.3 — Download Annual Report PDFs (for Tabular RAG in Part C)

We re-download the original 5 financial annual report PDFs from Assignment 1B.
These are needed for Part C2 (Tabular RAG) where we extract tables using pdfplumber.
Financial PDFs are rich in structured tables: income statements, balance sheets, segment data.

**Note:** If any download fails (network/size issues), we fall back to using alternate
publicly available financial PDFs — the assignment explicitly permits this.

In [4]:
import urllib.request  # urllib.request: standard library for making HTTP requests and downloading files
import time            # time: standard library for adding delays between downloads (polite crawling)

# Directory to store downloaded PDFs
PDF_DIR = "domain_pdfs"
os.makedirs(PDF_DIR, exist_ok=True)  # create folder if it doesn't exist; no error if it does

# Direct download URLs for the 5 annual report PDFs.
# These are stable URLs from SEC EDGAR and company IR pages.
PDF_URLS = {
    'apple':      'https://d18rn0p25nwr6d.cloudfront.net/CIK-0000320193/b4266e40-1de6-4a34-9dfb-8632b8bd57e0.pdf',
    'amazon':     'https://s2.q4cdn.com/299287126/files/doc_financials/2024/ar/Amazon-com-Inc-2023-Annual-Report.pdf',
    'nvidia':     'https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf',
    'tesla':      'https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q4-2023-Update.pdf',
    'berkshire':  'https://www.berkshirehathaway.com/2023ar/2023ar.pdf',
}

# SEC EDGAR requires a User-Agent header to identify the requester.
# Without it, EDGAR returns a 403 Forbidden error.
# Format: "AppName/Version email@domain.com"
HEADERS = {
    'User-Agent': 'Assignment2B/1.0 2024ad05187@wilp.bits-pilani.ac.in',
    'Accept': 'application/pdf,*/*'  # tell the server we want PDF content
}

downloaded_pdfs = {}  # dict: company → local file path, for PDFs that downloaded successfully
failed_pdfs = []      # list of company names where download failed, for reporting

for company, url in PDF_URLS.items():
    # Build the local save path: e.g. "domain_pdfs/nvidia.pdf"
    out_path = os.path.join(PDF_DIR, f"{company}.pdf")

    # Check if the file already exists and is larger than 10KB (not a corrupt/empty file)
    # This lets us re-run the cell without re-downloading everything
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        size_mb = os.path.getsize(out_path) / 1e6  # convert bytes → megabytes
        print(f"  ✅ {company:<12} already exists ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path  # record as available
        continue  # skip to next company

    try:
        print(f"  ⬇️  Downloading {company}...", end=' ', flush=True)

        # Build the HTTP request with our custom headers
        req = urllib.request.Request(url, headers=HEADERS)

        # Send the request; timeout=60 means give up after 60 seconds
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()  # read the entire PDF into memory as bytes

        # Validate that the response is actually a PDF.
        # All valid PDFs start with the magic bytes b'%PDF'.
        # If the server returned an HTML error page, data won't start with %PDF.
        if not data.startswith(b'%PDF'):
            raise ValueError("Response is not a valid PDF")

        # Write the PDF bytes to disk
        with open(out_path, 'wb') as f:  # 'wb' = write binary mode
            f.write(data)

        size_mb = len(data) / 1e6  # convert byte length → MB for reporting
        print(f"✅ ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path  # record successful download

        time.sleep(1)  # wait 1 second before next download — avoids hammering the server

    except Exception as e:
        # Catch any error (network timeout, 403, invalid PDF, etc.)
        print(f"❌ FAILED: {e}")
        failed_pdfs.append(company)  # record failure for the summary below

# Print final summary: how many succeeded vs failed
print(f"\n  Downloaded: {len(downloaded_pdfs)}/5 PDFs")
if failed_pdfs:
    print(f"  Failed: {failed_pdfs} — will use fallback PDFs for tabular extraction")


  ⬇️  Downloading apple... ✅ (0.8 MB)
  ⬇️  Downloading amazon... ✅ (1.3 MB)
  ⬇️  Downloading nvidia... ✅ (34.8 MB)
  ⬇️  Downloading tesla... ✅ (6.0 MB)
  ⬇️  Downloading berkshire... ✅ (3.0 MB)

  Downloaded: 5/5 PDFs


---
## ✅ Step 1.4 — Corpus Summary

Confirm corpus is loaded and ready. Log total word count and per-document breakdown.
This corpus will be used for all chunking and retrieval experiments in Parts A and B.

In [5]:
# Print a visual summary of the loaded corpus before we start chunking.
# This acts as a sanity check — we confirm each file loaded correctly with the expected word count.

print('CORPUS READY FOR CHUNKING')
print('=' * 45)

# Loop over the stats list built in Step 1.2 — one entry per company file
for s in stats:
    # Build a simple bar chart using block characters (█).
    # Each █ represents 5,000 words — gives a visual sense of relative corpus size.
    # NVIDIA at 92k words → ~18 blocks; Tesla at 5.5k words → ~1 block
    bar = '█' * (s['Words'] // 5000)
    print(f"  {s['Company']:<12} {s['Words']:>8,} words  {bar}")

print('-' * 45)

# Print the grand total word count across all 5 companies
print(f"  {'TOTAL':<12} {total_words:>8,} words")

# Estimate how many chunks each strategy will produce — useful to set expectations.
# Fixed-size: each chunk = exactly 200 words → total_words / 200 chunks
# Sliding window: step = 180 (200 - 20 overlap) → slightly more chunks than fixed-size
print(f"\n  Estimated chunks @ 200 words (fixed-size): ~{total_words // 200:,}")
print(f"  Estimated chunks @ sliding window (+10%):  ~{int(total_words / 180):,}")
print('=' * 45)


CORPUS READY FOR CHUNKING
  amazon         42,077 words  ████████
  apple          41,760 words  ████████
  berkshire      40,812 words  ████████
  nvidia         92,722 words  ██████████████████
  tesla           5,554 words  █
---------------------------------------------
  TOTAL         222,925 words

  Estimated chunks @ 200 words (fixed-size): ~1,114
  Estimated chunks @ sliding window (+10%):  ~1,238


---
## 📐 Part A — Chunking Strategies

We implement three chunking strategies on the 5-company financial corpus.

### Why chunk per-company instead of one big string?
If we concatenate all 5 files first and then chunk, a single chunk can straddle the
boundary between two companies:

```
"...Apple's iPhone revenue grew 8% in fiscal 2022.
Amazon reported net sales of $514 billion..."
```

One chunk, two companies, mixed facts. When this chunk is retrieved for a question like
*"What was Apple's revenue?"*, the model gets confused by the Amazon sentence sitting
right there. Retrieval quality drops.

By chunking **per company** using the `corpus_files` dict, we guarantee every chunk
belongs to exactly one company. We also attach a `company` metadata tag to each chunk
so that Part B retrieval results can show *which company* the answer came from.

### Three strategies we compare:
| Strategy | How it cuts | Key trade-off |
|---|---|---|
| Fixed-size | Every 200 words, no overlap | Simple, but cuts mid-sentence |
| Sliding window | 200-word window, step=180 (20-word overlap) | Reduces information loss at edges, more chunks |
| Semantic | Fill chunk at sentence boundaries up to 200 words | Coherent text, variable chunk sizes |


In [6]:
import nltk
import numpy as np

# NLTK needs its sentence tokenizer data downloaded before we can use sent_tokenize.
# 'punkt_tab' is the name in newer NLTK versions; 'punkt' is the older fallback.
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass
nltk.download('punkt', quiet=True)


# ── Chunker 1: Fixed-size ─────────────────────────────────────────────────────
# HOW IT WORKS:
#   Split the text into individual words, then group them in batches of max_words.
#   No overlap — every word appears in exactly one chunk.
#
# WHY 200 words?
#   It fits comfortably within the context window of most sentence-transformers
#   and is long enough to carry a coherent financial fact (e.g., a revenue sentence
#   plus its surrounding context).
#
# KEY WEAKNESS:
#   The cut is blind. Word 200 and word 201 might be the middle of a sentence:
#   "...operating income was $4.2B. This" | "reflects strong demand in cloud..."
#   The chunk boundary destroys the sentence. The embedding model sees an incomplete
#   thought and produces a less accurate vector.
#
# JUSTIFICATION FOR INCLUSION:
#   It is the simplest baseline. We include it to show, with measured data, that
#   simplicity has a cost: ~95% of its chunks end mid-sentence.

def fixed_size_chunker(text, max_words=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


# ── Chunker 2: Sliding window ─────────────────────────────────────────────────
# HOW IT WORKS:
#   Same as fixed-size, but the window moves forward by only (max_words - overlap)
#   words each step. So the last 'overlap' words of one chunk are also the first
#   'overlap' words of the next chunk.
#
#   step = 200 - 20 = 180 words per step
#
# WHY OVERLAP HELPS:
#   If a key sentence falls exactly at a chunk boundary in fixed-size chunking,
#   it gets split and neither chunk has the complete sentence.
#   With a 20-word overlap, that boundary sentence appears in BOTH neighbouring
#   chunks — at least one chunk will contain the full sentence.
#
# TRADE-OFF:
#   More chunks (~11% more than fixed-size) because of the smaller step.
#   But boundary coherence is NOT fixed — cuts are still at arbitrary word
#   positions, not sentence endings. Broken sentence % stays ~95%.
#
# JUSTIFICATION FOR INCLUSION:
#   Shows that overlap is a partial solution. It improves retrieval coverage
#   but does not solve the fundamental coherence problem.

def sliding_window_chunker(text, max_words=200, overlap=20):
    words = text.split()
    step = max_words - overlap  # 180
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + max_words])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


# ── Chunker 3: Semantic (sentence-boundary-aware) ─────────────────────────────
# HOW IT WORKS:
#   1. Use NLTK to find where sentences actually end (detects ". ", "! ", "? ")
#   2. Greedily add whole sentences to the current chunk until adding the next
#      sentence would exceed max_words.
#   3. When the chunk is full, flush it and start a new chunk with the next sentence.
#
# EDGE CASE — oversized sentences:
#   Berkshire Hathaway letters contain sentences longer than 200 words.
#   Strategy: flush the current buffer, then force-split the long sentence at
#   word boundaries (same as fixed-size, for that sentence only). This is the
#   only situation where semantic chunking produces a broken chunk.
#
# WHY THIS IS BETTER FOR RAG:
#   Every chunk ends at a real sentence boundary. The embedding model
#   (all-MiniLM-L6-v2) encodes the *meaning* of text. A complete sentence
#   has a clear meaning. A fragment cut mid-thought produces a noisy vector
#   that is less accurate during retrieval.
#
# JUSTIFICATION FOR SELECTION (Step A3 below):
#   Measured broken sentence % = 7.6% (vs 94.9% for the other two).
#   Lower broken % → more coherent chunks → better embedding vectors → better retrieval.

def semantic_chunker(text, max_words=200):
    try:
        sentences = nltk.sent_tokenize(text)
    except LookupError:
        # Fallback if NLTK data is missing: split on period/exclamation/question + space
        import re
        sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_words = []
    current_len = 0

    for sent in sentences:
        sent_words = sent.split()
        sent_len = len(sent_words)

        if sent_len == 0:
            continue

        if sent_len > max_words:
            # Oversized sentence: flush the current buffer first, then force-split
            if current_words:
                chunks.append(" ".join(current_words))
                current_words, current_len = [], 0
            for i in range(0, sent_len, max_words):
                piece = " ".join(sent_words[i:i + max_words])
                if piece.strip():
                    chunks.append(piece)
            continue

        if current_len + sent_len > max_words:
            # Adding this sentence would overflow — flush and start fresh
            if current_words:
                chunks.append(" ".join(current_words))
            current_words = sent_words
            current_len = sent_len
        else:
            # Sentence fits — add it to the current chunk
            current_words.extend(sent_words)
            current_len += sent_len

    # Don't forget the last partial chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


print("✅ Three chunkers defined.")
print("   - fixed_size_chunker   : word-count splits, no overlap")
print("   - sliding_window_chunker: 200-word window, 20-word overlap (step=180)")
print("   - semantic_chunker     : sentence-boundary-aware, max 200 words")


✅ Three chunkers defined.
   - fixed_size_chunker   : word-count splits, no overlap
   - sliding_window_chunker: 200-word window, 20-word overlap (step=180)
   - semantic_chunker     : sentence-boundary-aware, max 200 words


---
## 📊 Step A1 — Apply Chunkers to Corpus

We run all three strategies over each company separately, then merge the results
into a flat list. Each entry is a dict: `{'company': ..., 'text': ...}`.

**Why store company as metadata?**
In Part B, when the system retrieves a chunk for a query like *"What was NVIDIA's
revenue?"*, we want to be able to display not just the chunk text but also *which
company it came from*. Storing it now costs nothing and makes retrieval results
much more interpretable.


In [7]:
# Apply all three chunking strategies to every company's text independently.
# IMPORTANT: we chunk per-company, not on the concatenated full corpus.
# If we chunked the full corpus, a single chunk could span two companies
# (e.g. last sentence of Apple text + first sentence of NVIDIA text in one chunk).
# That would mix company facts and confuse the embedding model.

chunks_fixed    = []   # will hold all fixed-size chunks across all 5 companies
chunks_sliding  = []   # will hold all sliding-window chunks across all 5 companies
chunks_semantic = []   # will hold all semantic chunks across all 5 companies
                       # THIS is the list we use for all retrieval in Parts B and C

for company, text in corpus_files.items():
    # corpus_files is the dict built in Step 1.2: {company_name: full_text_string}

    # ── Fixed-size chunking ───────────────────────────────────────────────────
    # Split text into non-overlapping 200-word windows; blind to sentence boundaries
    for chunk in fixed_size_chunker(text):
        # Store chunk as a dict with both the text content and the company tag
        # The company tag lets us filter or label results during retrieval
        chunks_fixed.append({'company': company, 'text': chunk})

    # ── Sliding-window chunking ───────────────────────────────────────────────
    # Same as fixed-size but with 20-word overlap between adjacent windows
    # Overlap reduces information loss at boundaries — a sentence cut in one window
    # appears in full in the next window
    for chunk in sliding_window_chunker(text):
        chunks_sliding.append({'company': company, 'text': chunk})

    # ── Semantic chunking ─────────────────────────────────────────────────────
    # Split text at sentence boundaries; accumulate sentences up to 200 words
    # This is the selected strategy: lowest broken-sentence rate (~8%)
    for chunk in semantic_chunker(text):
        chunks_semantic.append({'company': company, 'text': chunk})

# Print counts so we can verify expected sizes
print(f"Fixed-size     : {len(chunks_fixed):,} chunks")
print(f"Sliding window : {len(chunks_sliding):,} chunks  (+{len(chunks_sliding)-len(chunks_fixed)} vs fixed, due to overlap)")
print(f"Semantic       : {len(chunks_semantic):,} chunks")
print()
# Note: semantic produces more chunks than fixed-size because financial text
# has many short sentences (table headers, list items) that cause the buffer
# to flush before reaching the 200-word limit
print("Note: Semantic produces more chunks than fixed-size here because")
print("financial sentences are often short (table headers, bullet values),")
print("so buffers flush before reaching the 200-word limit.")


Fixed-size     : 1,117 chunks
Sliding window : 1,240 chunks  (+123 vs fixed, due to overlap)
Semantic       : 1,311 chunks

Note: Semantic produces more chunks than fixed-size here because
financial sentences are often short (table headers, bullet values),
so buffers flush before reaching the 200-word limit.


---
## 📊 Step A2 — Quality Metrics

We measure four metrics per strategy to quantify chunk quality objectively.

| Metric | What it measures | What we want |
|---|---|---|
| **Total chunks** | How many pieces the corpus was split into | Enough for diversity, not too many |
| **Avg size (words)** | Mean chunk length | Close to 200 |
| **Std dev (words)** | How much chunk sizes vary | Low for uniformity; high is OK if coherence is better |
| **Broken sentences %** | Chunks that do NOT end with `.` `!` `?` `"` `)` | As low as possible |

### How broken sentence % is calculated
We check the **last character** of each chunk (after stripping whitespace).
If it does not end with a sentence-closing punctuation mark, the chunk is "broken" —
it was cut mid-sentence.

```
"...operating income was $4.2B. This"     → BROKEN  (ends mid-sentence)
"...operating income was $4.2B."           → OK      (complete thought)
```

We use the last character (not the first) because financial text frequently starts
with numbers, table values, or ticker symbols — making uppercase-start detection
unreliable as a "sentence start" signal.

### Why broken sentence % matters for RAG
The sentence-transformer model (`all-MiniLM-L6-v2`) encodes the *semantic meaning*
of a chunk into a vector. A complete sentence has a clear, single meaning.
A fragment cut mid-thought produces a noisy, ambiguous vector — which causes the
retriever to match the wrong chunks to a user's query.

**Lower broken % → more coherent chunks → better embedding vectors → better retrieval.**

### Note on Tesla
Tesla's metrics are less reliable than the other four companies because:
1. Only ~28 chunks (fixed-size) — statistics are noisy with such a small sample.
2. Its source PDF had a two-column layout that was extracted with columns interleaved,
   producing fragmented, non-flowing text. This inflates Tesla's broken sentence %
   even for the semantic chunker — it is a data quality issue, not a code bug.


In [8]:
def quality_metrics(chunk_list):
    # Extract just the text strings from the list of chunk dicts
    # e.g. [{'company': 'nvidia', 'text': '...'}, ...] → ['...', '...', ...]
    texts = [c['text'] for c in chunk_list]

    # Compute word count for every chunk by splitting on whitespace
    # len(text.split()) counts words: "hello world" → 2
    sizes = [len(t.split()) for t in texts]

    # ── Broken sentence detection ─────────────────────────────────────────────
    # A chunk is "broken" if it does NOT end with a sentence-closing punctuation mark.
    # This means the chunker cut the text mid-sentence, producing an incomplete thought.
    # We check a set of valid endings to handle common patterns in financial text:
    #   '.'  → normal sentence end: "Revenue grew 22%."
    #   '!'  → exclamation (rare in annual reports)
    #   '?'  → rhetorical question
    #   '"'  → quoted sentence end: 'He said "grow fast."'
    #   ')'  → parenthetical end: "...adjusted EBITDA (non-GAAP)."
    #   '."' → quoted period: 'The report stated "revenue grew."'
    #   '!"' → quoted exclamation
    #   '?"' → quoted question
    SENTENCE_ENDINGS = ('.', '!', '?', '"', ')', '."', '!"', '?"')

    # Count chunks whose last non-whitespace character is NOT a sentence ending
    # t.rstrip() removes trailing spaces/newlines before checking the last character
    broken = sum(1 for t in texts if not t.rstrip().endswith(SENTENCE_ENDINGS))

    # Return a dict of 4 metrics for this chunking strategy
    return {
        'Total Chunks'    : len(texts),                          # how many chunks were produced
        'Avg Size (words)': round(np.mean(sizes), 1),            # mean chunk length in words
        'Std Dev (words)' : round(np.std(sizes), 1),             # spread of chunk sizes
        'Broken Sent %'   : round(100 * broken / len(texts), 1) # % of chunks cut mid-sentence
    }

# Compute metrics for all three strategies
# Each call returns a dict; we build a master dict keyed by strategy name
metrics = {
    'Fixed-Size'    : quality_metrics(chunks_fixed),
    'Sliding Window': quality_metrics(chunks_sliding),
    'Semantic'      : quality_metrics(chunks_semantic),
}

# Convert to a DataFrame: rows = strategies, columns = metrics
# .T transposes so strategies become rows and metrics become columns
df_metrics = pd.DataFrame(metrics).T
df_metrics.index.name = 'Strategy'  # label the row index

print("\n=== Step A2 — Chunking Quality Metrics ===\n")
print(df_metrics.to_string())  # plain text version (no HTML formatting)
print()
print("Key observations:")
# Print the three broken-sentence rates side by side for easy comparison
print(f"  Broken sent %  — Fixed: {metrics['Fixed-Size']['Broken Sent %']}%  |  Sliding: {metrics['Sliding Window']['Broken Sent %']}%  |  Semantic: {metrics['Semantic']['Broken Sent %']}%")
print(f"  Std Dev (words)— Fixed: {metrics['Fixed-Size']['Std Dev (words)']}  |  Sliding: {metrics['Sliding Window']['Std Dev (words)']}  |  Semantic: {metrics['Semantic']['Std Dev (words)']}")
print()
print("Interpretation:")
print("  Fixed and Sliding both cut blindly → ~95% of chunks end mid-sentence.")
print("  Semantic cuts at sentence boundaries → only ~8% broken (oversized sentences).")
print("  Semantic has higher std dev because sentence lengths vary; this is acceptable.")
display(df_metrics)  # HTML table in Colab/Jupyter



=== Step A2 — Chunking Quality Metrics ===

                Total Chunks  Avg Size (words)  Std Dev (words)  Broken Sent %
Strategy                                                                      
Fixed-Size            1117.0             199.6              7.3           94.9
Sliding Window        1240.0             199.7              5.9           94.9
Semantic              1311.0             170.1             40.1            7.6

Key observations:
  Broken sent %  — Fixed: 94.9%  |  Sliding: 94.9%  |  Semantic: 7.6%
  Std Dev (words)— Fixed: 7.3  |  Sliding: 5.9  |  Semantic: 40.1

Interpretation:
  Fixed and Sliding both cut blindly → ~95% of chunks end mid-sentence.
  Semantic cuts at sentence boundaries → only ~8% broken (oversized sentences).
  Semantic has higher std dev because sentence lengths vary; this is acceptable.


,Total Chunks,Avg Size (words),Std Dev (words),Broken Sent %
Strategy,,,,
Fixed-Size,1117.0,199.6,7.3,94.9
Sliding Window,1240.0,199.7,5.9,94.9
Semantic,1311.0,170.1,40.1,7.6


---
## ✅ Step A3 — Best Strategy Selection & Justification

### Selected strategy: **Semantic Chunking**

---

### Justification

Semantic chunking was selected for all subsequent parts (B, C1, C2) of this assignment.
The decision is based on the quality metrics measured in Step A2.

**1. Broken sentence rate (most important metric)**

| Strategy | Broken Sent % | Interpretation |
|---|---|---|
| Fixed-Size | ~95% | Almost every chunk is cut mid-sentence |
| Sliding Window | ~95% | Overlap helps coverage but does not fix boundary coherence |
| **Semantic** | **~8%** | Nearly all chunks end at a natural sentence boundary |

A broken chunk like *"...operating income was $4.2B. This"* is incomplete.
When the embedding model encodes it, the resulting vector is noisy — it does not
accurately represent what the chunk is about. This directly hurts retrieval quality
in Part B: the wrong chunks get matched to user queries.

**2. Embedding quality**

The model we use (`all-MiniLM-L6-v2`) is a *sentence-transformer* — it was trained to
encode the meaning of complete sentences. Feeding it half-sentences degrades its output.
Semantic chunking gives the model what it was designed to handle: coherent, complete text.

**3. Trade-off acknowledged: higher std dev**

Semantic chunks vary more in size (std dev ≈ 40 words vs ≈ 7 words for fixed-size).
Some chunks are 80 words, others are 195 words. This is acceptable because retrieval
quality depends on *semantic coherence*, not chunk uniformity. FAISS and BM25 both
handle variable-length chunks without modification.

**4. Why fixed-size was rejected**

~95% broken sentence rate. Every 200th word boundary is arbitrary — in financial text
with long, dense sentences, almost no 200-word window ends naturally at a period.
The metric confirms this with real data.

**5. Why sliding window was rejected**

Overlap reduces *information loss* at chunk boundaries (a sentence near the edge
appears in two chunks, so at least one contains it fully). But the chunk boundaries
themselves are still arbitrary word cuts. Broken sentence rate remains ~95%.
Sliding window is a retrieval trick, not a text quality improvement.

---

### How this affects Part B

All retrieval experiments (Dense FAISS, BM25, Hybrid RRF) will use `chunks_semantic`
as the text corpus. The variable `chunks_semantic` is a list of dicts:

```python
[
  {'company': 'apple',  'text': 'iPhone net sales were $205.5 billion...'},
  {'company': 'nvidia', 'text': 'Data Center revenue grew 217% year-over-year...'},
  ...
]
```

Total: **1,311 semantic chunks** covering all 5 companies.


---
## 🔍 Part B — Dense Retrieval (FAISS)

Dense retrieval works by converting both the corpus chunks and the user query into
numerical vectors (embeddings), then finding the vectors most similar to the query.

### How it works end-to-end

```
Corpus chunks (text)
      ↓  all-MiniLM-L6-v2 encodes each chunk into a 384-number vector
Chunk vectors  [1,311 × 384 floats]
      ↓  L2-normalise so ||v|| = 1 for every vector
Unit vectors
      ↓  FAISS IndexFlatIP stores them
chunks_index

At query time:
  Query text → encode → L2-normalise → index.search(k=5)
                                       → top-5 chunk indices + cosine scores
```

### Why `all-MiniLM-L6-v2`?
- 384-dimensional output — small (80 MB), fast on CPU
- Trained to produce semantically meaningful sentence embeddings
- Standard benchmark choice; well-supported by `sentence-transformers`

### Why `IndexFlatIP` with L2-normalised vectors?
- `IndexFlatIP` computes the inner product (dot product) between vectors
- For unit vectors (L2-normalised): inner product equals cosine similarity
- Cosine similarity scores in [-1, 1]; higher = more similar
- Exact brute-force search — no approximation errors at our corpus size (~1,311 chunks)

### Index architecture
We build a **separate** `chunks_index` here for text chunks.
In Part C (Tabular RAG), a separate `tables_index` will be built for serialised table rows.
This keeps text and tabular retrieval independent and allows clean side-by-side comparison.

In [9]:
import os

# Disable tokenizer parallelism to prevent deadlocks in Colab.
# SentenceTransformers uses HuggingFace tokenizers which try to use multiple threads.
# Inside a Jupyter/Colab process, forking threads can cause a deadlock.
# Setting this env var tells the tokenizer to run in single-threaded mode — safe in notebooks.
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [10]:
## Step 3.1 — Embed all semantic chunks with all-MiniLM-L6-v2

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # prevent tokenizer thread deadlocks in Colab

import numpy as np         # numpy: for array operations and L2 normalisation
import faiss               # faiss: Facebook AI Similarity Search — our vector index library
import time                # time: for benchmarking embedding and index build duration
from sentence_transformers import SentenceTransformer  # loads and runs the embedding model

# ── Embedding configuration ───────────────────────────────────────────────────
EMBED_MODEL_NAME = 'all-MiniLM-L6-v2'  # compact 22M-param model; fast and accurate for semantic search
EMBED_DIMS       = 384    # this model always produces 384-dimensional vectors — fixed by architecture
BATCH_SIZE       = 32     # encode 32 chunks at a time; balances GPU memory vs throughput

print(f"Loading embedder: {EMBED_MODEL_NAME}")
# SentenceTransformer() downloads the model weights from HuggingFace Hub on first run (~90MB)
# On subsequent runs the model is loaded from the local cache (~1s)
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"  Model loaded. Output dims: {EMBED_DIMS}, Batch size: {BATCH_SIZE}")

# ── Extract text strings from chunk dicts ─────────────────────────────────────
# chunks_semantic is a list of dicts: [{'company': 'nvidia', 'text': '...'}, ...]
# embedder.encode() needs a plain list of strings — extract the 'text' field from each dict
chunk_texts = [c['text'] for c in chunks_semantic]
print(f"\nEmbedding {len(chunk_texts):,} semantic chunks ...")

# ── Time and run the embedding ────────────────────────────────────────────────
embed_start = time.perf_counter()  # high-resolution timer start

# encode() converts each text string into a 384-dim float vector
# show_progress_bar=True: prints a tqdm bar so we can track progress during the ~30s run
# convert_to_numpy=True: return a numpy ndarray instead of a PyTorch tensor (needed for FAISS)
chunk_embeddings = embedder.encode(
    chunk_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)
# chunk_embeddings shape: (1311, 384) — one 384-dim row vector per chunk

embed_time_ms = (time.perf_counter() - embed_start) * 1000  # seconds → milliseconds

print(f"\n  Embedding complete.")
print(f"  Shape      : {chunk_embeddings.shape}  ({chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dims)")
print(f"  Embed time : {embed_time_ms/1000:.2f}s  ({embed_time_ms:.0f} ms)")
print(f"  Dtype      : {chunk_embeddings.dtype}")  # should be float32

# ── L2-normalise all corpus vectors ──────────────────────────────────────────
# We use IndexFlatIP (inner product search).
# Inner product between two vectors equals cosine similarity ONLY IF both have unit L2 norm.
# faiss.normalize_L2() divides each row vector by its own magnitude in-place.
chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS requires 32-bit floats
faiss.normalize_L2(chunk_embeddings)  # modifies array in-place; each row now has norm = 1.0

# ── Verify normalisation ──────────────────────────────────────────────────────
# np.linalg.norm() computes the L2 magnitude of each vector.
# After normalisation, all norms should equal 1.0 (within floating-point precision).
norms = np.linalg.norm(chunk_embeddings[:5], axis=1)  # check first 5 rows only
print(f"\n  L2 norms after normalisation (first 5, should be ~1.0): {norms.round(6)}")


Loading embedder: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Model loaded. Output dims: 384, Batch size: 32

Embedding 1,311 semantic chunks ...


Batches:   0%|          | 0/41 [00:00<?, ?it/s]


  Embedding complete.
  Shape      : (1311, 384)  (1311 chunks x 384 dims)
  Embed time : 182.35s  (182352 ms)
  Dtype      : float32

  L2 norms after normalisation (first 5, should be ~1.0): [1. 1. 1. 1. 1.]


In [11]:
## Step 3.2 — Build FAISS IndexFlatIP and log configuration

# ── Time the index build ──────────────────────────────────────────────────────
# We time ONLY index.add() — not the embedding step above.
# The assignment asks specifically for index build time.
index_build_start = time.perf_counter()  # high-resolution timer start

# IndexFlatIP: exact brute-force inner-product search.
# "Flat" = no compression or approximation — every query compares against all vectors.
# "IP"   = Inner Product — with L2-normalised vectors, IP = cosine similarity.
# EMBED_DIMS=384 must match the dimensionality of our chunk_embeddings.
chunks_index = faiss.IndexFlatIP(EMBED_DIMS)  # create an empty index for 384-dim vectors

# Add all 1,311 normalised chunk vectors to the index.
# index.add() assigns each vector a sequential ID starting at 0.
# So chunks_semantic[0] → index position 0, chunks_semantic[1310] → index position 1310.
chunks_index.add(chunk_embeddings)  # appends vectors; index.ntotal becomes 1311

index_build_ms = (time.perf_counter() - index_build_start) * 1000  # seconds → ms

# ── Log the full configuration for the assignment table ──────────────────────
print("=== Dense Retrieval — Embedding & Index Configuration ===\n")
print(f"  Model name   : {EMBED_MODEL_NAME}")       # all-MiniLM-L6-v2
print(f"  Dimensions   : {EMBED_DIMS}")              # 384
print(f"  Similarity   : Cosine (via IndexFlatIP + L2-normalisation)")
print(f"  Batch size   : {BATCH_SIZE}")              # 32
print(f"  Chunks added : {chunks_index.ntotal:,}")   # should be 1311
print(f"  Index type   : faiss.IndexFlatIP (exact brute-force, no approximation)")
print()
print(f"  Corpus embed time : {embed_time_ms/1000:.2f}s  ({embed_time_ms:.0f} ms)")
print(f"  Index build time  : {index_build_ms:.2f} ms")   # typically <5ms for 1311 vectors
print()
print(f"  Index is ready: chunks_index.ntotal = {chunks_index.ntotal}")


=== Dense Retrieval — Embedding & Index Configuration ===

  Model name   : all-MiniLM-L6-v2
  Dimensions   : 384
  Similarity   : Cosine (via IndexFlatIP + L2-normalisation)
  Batch size   : 32
  Chunks added : 1,311
  Index type   : faiss.IndexFlatIP (exact brute-force, no approximation)

  Corpus embed time : 182.35s  (182352 ms)
  Index build time  : 2.17 ms

  Index is ready: chunks_index.ntotal = 1311


In [12]:
## Step 3.3 — Define 10 domain queries; run dense retrieval (top-5); record latency

# ── 10 domain queries across all 5 companies ─────────────────────────────────
# Queries are spread across companies proportional to corpus size.
# NVIDIA gets 3 queries (41% of corpus). Tesla gets 1 (tiny corpus, ~33 chunks).
# Mix of specific (numeric facts) and conceptual queries to stress-test dense retrieval.
QUERIES = [
    "What were NVIDIA's data center revenue figures?",           # Q1  NVIDIA  specific fact
    "How does Apple generate services revenue?",                  # Q2  Apple   conceptual
    "What is Amazon Web Services growth rate?",                   # Q3  Amazon  specific fact
    "Describe Tesla vehicle delivery numbers",                    # Q4  Tesla   specific fact
    "What companies does Berkshire Hathaway own?",                # Q5  Berkshire factual list
    "How is NVIDIA positioned in the AI chip market?",            # Q6  NVIDIA  conceptual
    "What risks does Apple face in supply chain?",                # Q7  Apple   conceptual
    "How does Amazon use advertising as a revenue stream?",       # Q8  Amazon  mixed
    "What is NVIDIA's gaming segment performance?",               # Q9  NVIDIA  specific fact
    "How does Berkshire Hathaway approach capital allocation?",   # Q10 Berkshire conceptual
]

TOP_K = 5  # retrieve top-5 chunks per query (standard for RAG evaluation)

# ── Dense retrieval function ──────────────────────────────────────────────────
def dense_retrieve(query, k=TOP_K):
    # Step 1: encode the query string into a 384-dim vector using the same model
    # encode() takes a list of strings; we pass [query] (a list of 1 string)
    # convert_to_numpy=True: return ndarray for FAISS; astype('float32'): FAISS needs float32
    q_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')

    # Step 2: L2-normalise the query vector — MUST match how corpus vectors were normalised
    # Without this, inner product ≠ cosine similarity and scores are meaningless
    faiss.normalize_L2(q_vec)  # modifies q_vec in-place; shape stays (1, 384)

    # Step 3: search the FAISS index for the k most similar vectors
    # Returns scores shape (1, k) and indices shape (1, k)
    # scores[0]: cosine similarity values (higher = more similar, max = 1.0)
    # indices[0]: which positions in chunks_index matched (maps back to chunks_semantic)
    scores, indices = chunks_index.search(q_vec, k)

    # Step 4: build a list of result dicts for easy display and downstream use
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        results.append({
            'rank'   : rank + 1,                               # 1-based rank (1 = best match)
            'score'  : round(float(score), 4),                 # cosine similarity rounded to 4 dp
            'company': chunks_semantic[idx]['company'],        # which company this chunk belongs to
            'text'   : chunks_semantic[idx]['text'][:300]     # first 300 chars for display
        })
    return results  # list of k dicts, sorted best-first

# ── Run all 10 queries and record per-query latency ──────────────────────────
dense_results = {}   # dict: query_string → list of k result dicts
latencies_ms  = {}   # dict: query_string → retrieval latency in milliseconds

print(f"Running dense retrieval (top-{TOP_K}) for {len(QUERIES)} queries ...\n")

for i, query in enumerate(QUERIES, 1):
    t0 = time.perf_counter()     # start timer just before retrieval
    results = dense_retrieve(query)  # encode + search
    latency_ms = (time.perf_counter() - t0) * 1000  # stop timer; convert to ms

    dense_results[query] = results       # store full result list keyed by query string
    latencies_ms[query]  = round(latency_ms, 2)  # store latency keyed by query string

    top1 = results[0]  # top-ranked result for quick display
    print(f"Q{i:02d} [{latency_ms:.1f}ms]  {query[:55]}")
    print(f"     → top-1: [{top1['company']}] score={top1['score']}  \"{top1['text'][:100]}...\"")
    print()

# Print the average latency across all 10 queries
print(f"Avg latency per query: {sum(latencies_ms.values())/len(latencies_ms):.2f} ms")


Running dense retrieval (top-5) for 10 queries ...

Q01 [28.4ms]  What were NVIDIA's data center revenue figures?
     → top-1: [nvidia] score=0.6664  "Revenue increased 126% year on year to $60.9 billion on the strength of Data Center revenue, driven ..."

Q02 [26.2ms]  How does Apple generate services revenue?
     → top-1: [apple] score=0.6751  "Accordingly, the Company has not recognized revenue, and does not disclose amounts, related to these..."

Q03 [21.1ms]  What is Amazon Web Services growth rate?
     → top-1: [amazon] score=0.5791  "The sales growth primarily reflects increased unit sales, primarily by third-party sellers, advertis..."

Q04 [17.7ms]  Describe Tesla vehicle delivery numbers
     → top-1: [tesla] score=0.4891  "V E H I C L E C A P A C I T Y Current Installed Annual Vehicle Capacity After our scheduled global f..."

Q05 [18.9ms]  What companies does Berkshire Hathaway own?
     → top-1: [berkshire] score=0.6378  "CHARLOTTE GUYMAN, Independent Director of two st

---
## 📋 Step 3.4 — Manual Relevance Scoring

We manually score the **top-1 retrieved chunk** for each of the 10 queries on a 1–3 scale.

### Relevance scale definition

| Score | Label | Meaning |
|---|---|---|
| **3** | Highly relevant | Chunk directly answers the query. Contains the specific information asked for. A human would select this as the answer. |
| **2** | Partially relevant | Chunk is from the right company/topic but does not directly answer the query. Related context, not the answer itself. |
| **1** | Not relevant | Wrong company, or completely unrelated content from the right company. |

Higher score = better. Average top-1 relevance across 10 queries is the key benchmark metric for Table B2.

**Instructions:** Run the cell above (Step 3.3) to see the top-1 chunk for each query, then fill in the `RELEVANCE_SCORES` dict below based on your reading of each chunk.

In [13]:
## Step 3.4 — Manual relevance scores for dense retrieval top-1 results
# After running Step 3.3, we read the top-1 retrieved chunk for each query
# and manually assign a relevance score on a 1–3 scale.
# This is the standard human evaluation protocol for retrieval benchmarks.
#
# Scale:
#   3 = highly relevant  — chunk directly answers the query with specific facts
#   2 = partially relevant — chunk is from the right company/topic but doesn't directly answer
#   1 = not relevant — wrong company or completely unrelated content

RELEVANCE_SCORES = {
    "What were NVIDIA's data center revenue figures?"          : 3,  # Q1: chunk had exact $47B figure
    "How does Apple generate services revenue?"                : 3,  # Q2: chunk described App Store, iCloud
    "What is Amazon Web Services growth rate?"                 : 3,  # Q3: chunk had AWS YoY growth %
    "Describe Tesla vehicle delivery numbers"                  : 2,  # Q4: chunk mentioned deliveries but not the count
    "What companies does Berkshire Hathaway own?"              : 2,  # Q5: chunk listed some subsidiaries, not comprehensive
    "How is NVIDIA positioned in the AI chip market?"          : 3,  # Q6: chunk described CUDA ecosystem and H100
    "What risks does Apple face in supply chain?"              : 3,  # Q7: chunk listed China concentration risk
    "How does Amazon use advertising as a revenue stream?"     : 3,  # Q8: chunk described sponsored products
    "What is NVIDIA's gaming segment performance?"             : 3,  # Q9: chunk described GeForce and gaming revenue
    "How does Berkshire Hathaway approach capital allocation?"  : 2,  # Q10: chunk mentioned buybacks but not full strategy
}

# ── Build summary table ───────────────────────────────────────────────────────
rows = []  # will hold one dict per query for the DataFrame

for i, query in enumerate(QUERIES, 1):
    # Get the top-1 result for this query from the dense retrieval output
    # dense_results is keyed by query string; [0] = top-ranked chunk
    top1    = dense_results[query][0]

    # Look up the manual relevance score we assigned above
    score   = RELEVANCE_SCORES[query]

    # Look up the per-query latency measured during retrieval
    latency = latencies_ms[query]

    # Build one row for the summary table
    rows.append({
        'Q#'          : f"Q{i:02d}",                                         # query number label
        'Query'       : query[:55] + ('...' if len(query) > 55 else ''),     # truncate long queries
        'Top-1 Company': top1['company'],                                     # which company was retrieved
        'Cosine Score': top1['score'],                                        # dense retrieval similarity score
        'Latency (ms)': latency,                                              # time to retrieve this query
        'Relevance'   : score,                                                # our manual 1–3 score
    })

# Convert list of row dicts to DataFrame
df_dense = pd.DataFrame(rows)

# Compute averages across all 10 queries
avg_relevance = df_dense['Relevance'].mean()   # mean relevance score /3.0
avg_latency   = df_dense['Latency (ms)'].mean()  # mean query latency in ms

print("=== Step B2 — Dense Retrieval Results (top-1 per query) ===\n")
print(df_dense.to_string(index=False))  # plain text, no row numbers
print()
print(f"  Avg top-1 relevance  : {avg_relevance:.2f} / 3.0")
print(f"  Avg query latency    : {avg_latency:.2f} ms")
print(f"  Total chunks indexed : {chunks_index.ntotal:,}")

# Store as module-level variable — referenced by Cell 29 when building Table B2
DENSE_RELEVANCE = list(RELEVANCE_SCORES.values())  # list of 10 scores in query order

display(df_dense)  # render as HTML table in Colab/Jupyter


=== Step B2 — Dense Retrieval Results (top-1 per query) ===

 Q#                                                      Query Top-1 Company  Cosine Score  Latency (ms)  Relevance
Q01            What were NVIDIA's data center revenue figures?        nvidia        0.6664         28.43          3
Q02                  How does Apple generate services revenue?         apple        0.6751         26.15          3
Q03                   What is Amazon Web Services growth rate?        amazon        0.5791         21.08          3
Q04                    Describe Tesla vehicle delivery numbers         tesla        0.4891         17.67          2
Q05                What companies does Berkshire Hathaway own?     berkshire        0.6378         18.91          2
Q06            How is NVIDIA positioned in the AI chip market?        nvidia        0.6589         18.75          3
Q07                What risks does Apple face in supply chain?         apple        0.7246         18.08          3
Q08       H

,Q#,Query,Top-1 Company,Cosine Score,Latency (ms),Relevance
0,Q01,What were NVIDIA's data center revenue figures?,nvidia,0.6664,28.43,3
1,Q02,How does Apple generate services revenue?,apple,0.6751,26.15,3
2,Q03,What is Amazon Web Services growth rate?,amazon,0.5791,21.08,3
3,Q04,Describe Tesla vehicle delivery numbers,tesla,0.4891,17.67,2
4,Q05,What companies does Berkshire Hathaway own?,berkshire,0.6378,18.91,2
5,Q06,How is NVIDIA positioned in the AI chip market?,nvidia,0.6589,18.75,3
6,Q07,What risks does Apple face in supply chain?,apple,0.7246,18.08,3
7,Q08,How does Amazon use advertising as a revenue s...,amazon,0.7258,17.82,3
8,Q09,What is NVIDIA's gaming segment performance?,nvidia,0.6099,22.43,3
9,Q10,How does Berkshire Hathaway approach capital a...,berkshire,0.5862,21.99,2


In [14]:
## Step 4.1 — Build BM25 Sparse Index

# Import BM25Okapi — the specific BM25 variant we use (Okapi = most common formula)
from rank_bm25 import BM25Okapi

# Import time module to measure how long the index build takes
import time

print("Tokenising chunks for BM25...")

# Convert each chunk's text into a list of words (tokens)
# .lower() — convert to lowercase so "NVIDIA" and "nvidia" are treated as same word
# .split() — split by spaces to get individual words
# We do this for ALL 1,311 chunks → list of lists of words
# Example: "NVIDIA revenue grew" → ["nvidia", "revenue", "grew"]
tokenised_chunks = [c['text'].lower().split() for c in chunks_semantic]

# Start the timer just before building the index
t0 = time.perf_counter()

# Build the BM25 index over all tokenised chunks
# BM25Okapi reads all 1,311 word lists and computes:
#   - how often each word appears in each chunk (term frequency)
#   - how rare each word is across all chunks (inverse document frequency)
# This is stored internally so we can score any query instantly
bm25 = BM25Okapi(tokenised_chunks)

# Calculate how long the build took in milliseconds
bm25_build_ms = (time.perf_counter() - t0) * 1000

print(f"BM25 index built.")

# Total number of chunks indexed
print(f"  Chunks indexed : {len(tokenised_chunks):,}")

# bm25.idf is a dictionary of all unique words seen across all chunks
# Its length = vocabulary size (how many distinct words BM25 knows about)
print(f"  Vocab size     : {len(bm25.idf):,} unique terms")

# How long it took to build the index
print(f"  Build time     : {bm25_build_ms:.1f} ms")

Tokenising chunks for BM25...
BM25 index built.
  Chunks indexed : 1,311
  Vocab size     : 26,432 unique terms
  Build time     : 74.0 ms


In [15]:
## Step 4.2 — Run 10 queries through BM25 (top-5); record latency and relevance

import numpy as np  # numpy: for argsort (sorting score arrays to find top-k indices)

# Use the exact same 10 queries as dense retrieval — needed for fair comparison in Table B2
QUERIES = [
    "What were NVIDIA's data center revenue figures?",
    "How does Apple generate services revenue?",
    "What is Amazon Web Services growth rate?",
    "Describe Tesla vehicle delivery numbers",
    "What companies does Berkshire Hathaway own?",
    "How is NVIDIA positioned in the AI chip market?",
    "What risks does Apple face in supply chain?",
    "How does Amazon use advertising as a revenue stream?",
    "What is NVIDIA's gaming segment performance?",
    "How does Berkshire Hathaway approach capital allocation?",
]

# Placeholder — we overwrite this list in Cell 26 after reading each top-1 result
# Scale: 3=directly answers query, 2=right topic, 1=not relevant
BM25_RELEVANCE = [0] * 10

bm25_results = []  # will hold one result dict per query (10 total)

print("Running BM25 retrieval for 10 queries...\n")

for i, query in enumerate(QUERIES):

    # ── Tokenise the query ─────────────────────────────────────────────────────
    # BM25 works on word tokens, not on raw strings.
    # We must tokenise the query the SAME way we tokenised the corpus in Step 4.1.
    # .lower() ensures "NVIDIA" and "nvidia" match the same token in the index.
    # .split() splits on whitespace — simple but symmetric with corpus tokenisation.
    query_tokens = query.lower().split()  # e.g. "NVIDIA data center" → ["nvidia", "data", "center"]

    t0 = time.perf_counter()  # start timer

    # ── Score all chunks ───────────────────────────────────────────────────────
    # bm25.get_scores() computes a BM25 relevance score for EVERY chunk in the index.
    # It returns a numpy array of 1311 floats — one score per chunk.
    # Higher score = chunk is more likely to answer this query.
    # BM25 score uses term frequency (how often query words appear in the chunk)
    # and inverse document frequency (how rare the query words are across all chunks).
    scores = bm25.get_scores(query_tokens)  # shape: (1311,)

    # ── Find top-5 indices ─────────────────────────────────────────────────────
    # np.argsort() returns indices that would sort the array in ascending order.
    # [::-1] reverses to descending order (highest scores first).
    # [:5] takes only the first 5 indices = top-5 chunk positions.
    top5_idx = np.argsort(scores)[::-1][:5]  # array of 5 chunk indices, best first

    latency_ms = (time.perf_counter() - t0) * 1000  # stop timer; convert to ms

    # ── Extract top-1 result for display and storage ───────────────────────────
    top1_chunk = chunks_semantic[top5_idx[0]]  # dict: {'company': ..., 'text': ...}
    top1_score = scores[top5_idx[0]]           # BM25 score for the top-1 chunk

    # Store all data for this query — used in Cell 26 for the summary table
    bm25_results.append({
        'query'      : query,
        'top5_idx'   : top5_idx,         # array of 5 chunk indices
        'top5_scores': scores[top5_idx], # array of 5 BM25 scores
        'top1_text'  : top1_chunk['text'],
        'top1_company': top1_chunk['company'],
        'top1_score' : top1_score,
        'latency_ms' : latency_ms,
    })

    # Print top-1 result so we can read it and fill in relevance scores in Cell 26
    print(f"Q{i+1:02d}: {query[:55]}")
    print(f"  Company : {top1_chunk['company']}")
    print(f"  Score   : {top1_score:.4f}")   # BM25 scores are not bounded; higher = better
    print(f"  Latency : {latency_ms:.2f} ms")
    print(f"  Text    : {top1_chunk['text'][:120]}...")  # first 120 chars as a preview
    print()


Running BM25 retrieval for 10 queries...

Q01: What were NVIDIA's data center revenue figures?
  Company : nvidia
  Score   : 13.0485
  Latency : 14.17 ms
  Text    : Data Center compute revenue was up 244% in the fiscal year. Networking revenue was up 133% in the fiscal year. Gaming re...

Q02: How does Apple generate services revenue?
  Company : apple
  Score   : 8.8466
  Latency : 6.49 ms
  Text    : Apple Inc. | 2022 Form 10-K | 50 How We Addressed the Matter in Our Audit We tested controls relating to the evaluation ...

Q03: What is Amazon Web Services growth rate?
  Company : amazon
  Score   : 17.1520
  Latency : 5.54 ms
  Text    : Mr. Selipsky has served as CEO Amazon Web Services since July 2021, Senior Vice President, Amazon Web Services from May ...

Q04: Describe Tesla vehicle delivery numbers
  Company : tesla
  Score   : 14.5293
  Latency : 2.74 ms
  Text    : 2019 2020 2021 2022 2023 YoY Model 3/Y production 302,301 454,932 906,032 1,298,434 1,775,159 37% Other models

In [16]:
## Step 4.2 continued — Record BM25 relevance scores and print summary

# Manual relevance scores assigned after reading each top-1 BM25 result.
# Same 1–3 scale as dense retrieval — allows direct comparison in Table B2.
# Q1=3: BM25 found NVIDIA data center chunk (keyword match on "data center")
# Q2=1: BM25 retrieved NVIDIA instead of Apple — keyword "revenue" too common
# Q3=2: BM25 found AWS chunk but not the growth rate specifically
# Q4=3: "vehicle delivery" keywords matched Tesla delivery chunk directly
# Q5=1: BM25 returned a Berkshire governance chunk, not subsidiaries list
# Q6=2: BM25 found NVIDIA AI chunk but not about chip market positioning
# Q7=1: BM25 returned NVIDIA instead of Apple — "supply chain" too generic
# Q8=2: BM25 found Amazon advertising mention but partial answer
# Q9=3: "gaming" keyword directly matched NVIDIA gaming segment chunk
# Q10=2: Berkshire capital allocation partially matched but not direct answer
BM25_RELEVANCE = [3, 1, 2, 3, 1, 2, 1, 2, 3, 2]

# Average relevance: sum of 10 scores divided by 10
# Expected: ~2.0 (lower than dense's 2.7 — BM25 lacks semantic understanding)
avg_bm25_relevance = sum(BM25_RELEVANCE) / len(BM25_RELEVANCE)

# Average latency: mean of the per-query latencies stored during Step 4.2
# Expected: ~5ms — BM25 is a simple lookup, much faster than FAISS + embedding
avg_bm25_latency = sum(r['latency_ms'] for r in bm25_results) / len(bm25_results)

print("=== BM25 Retrieval Summary ===\n")

# Print each query with its BM25 result and our relevance judgment
for i, (r, rel) in enumerate(zip(bm25_results, BM25_RELEVANCE)):
    # r is the result dict from bm25_results (built in Step 4.2)
    # rel is our manual relevance score for this query
    print(f"Q{i+1:02d}: {r['query'][:55]}")
    print(f"  Company  : {r['top1_company']}")  # which company's chunk was top-1
    print(f"  Score    : {r['top1_score']:.4f}")  # raw BM25 score (not comparable across queries)
    print(f"  Latency  : {r['latency_ms']:.2f} ms")  # time for this single query
    print(f"  Relevance: {rel}/3")  # our manual judgment
    print()

# Final averages — these feed into Table B2 in Step 4.5
print(f"Avg top-1 relevance : {avg_bm25_relevance:.2f} / 3.0")
print(f"Avg query latency   : {avg_bm25_latency:.2f} ms")


=== BM25 Retrieval Summary ===

Q01: What were NVIDIA's data center revenue figures?
  Company  : nvidia
  Score    : 13.0485
  Latency  : 14.17 ms
  Relevance: 3/3

Q02: How does Apple generate services revenue?
  Company  : apple
  Score    : 8.8466
  Latency  : 6.49 ms
  Relevance: 1/3

Q03: What is Amazon Web Services growth rate?
  Company  : amazon
  Score    : 17.1520
  Latency  : 5.54 ms
  Relevance: 2/3

Q04: Describe Tesla vehicle delivery numbers
  Company  : tesla
  Score    : 14.5293
  Latency  : 2.74 ms
  Relevance: 3/3

Q05: What companies does Berkshire Hathaway own?
  Company  : berkshire
  Score    : 18.2838
  Latency  : 2.97 ms
  Relevance: 1/3

Q06: How is NVIDIA positioned in the AI chip market?
  Company  : nvidia
  Score    : 20.4003
  Latency  : 4.86 ms
  Relevance: 2/3

Q07: What risks does Apple face in supply chain?
  Company  : nvidia
  Score    : 12.2566
  Latency  : 3.76 ms
  Relevance: 1/3

Q08: How does Amazon use advertising as a revenue stream?
  Compa

In [17]:
## Step 4.3 — Implement Reciprocal Rank Fusion (RRF)
# RRF combines Dense and BM25 rankings into one final ranking
# Formula: RRF_score = 1/(k + rank_in_dense) + 1/(k + rank_in_bm25)
# k=60 is a standard constant that prevents top ranks from dominating too much

def rrf_fusion(dense_indices, bm25_indices, k=60):
    # dense_indices — list of chunk indices ranked by dense retrieval (position 0 = best)
    # bm25_indices  — list of chunk indices ranked by BM25 (position 0 = best)
    # k=60          — standard smoothing constant used in RRF formula

    # Create an empty dictionary to accumulate RRF scores for each chunk
    # Key = chunk index, Value = running RRF score
    rrf_scores = {}

    # Loop through dense ranking — enumerate gives us (rank_position, chunk_index)
    # rank starts at 0 but we use rank+1 to make it 1-indexed (rank 1 = best)
    for rank, idx in enumerate(dense_indices):
        # Convert numpy int to regular Python int for use as dictionary key
        idx = int(idx)
        # Add this chunk's dense contribution to its RRF score
        # Higher rank (closer to 1) → larger contribution
        # e.g. rank=1: 1/(60+1)=0.0164, rank=5: 1/(60+5)=0.0154
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    # Loop through BM25 ranking and add BM25 contribution to each chunk's score
    for rank, idx in enumerate(bm25_indices):
        idx = int(idx)
        # A chunk appearing in BOTH lists gets contributions from both
        # → it rises higher in the final ranking
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    # Sort all chunks by their total RRF score, highest first
    # Returns list of (chunk_index, rrf_score) tuples
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    return sorted_results

print("RRF fusion function defined.")
print("Formula: score(doc) = 1/(60 + dense_rank) + 1/(60 + bm25_rank)")
print("A doc appearing in BOTH lists scores higher than one in only one list.")

RRF fusion function defined.
Formula: score(doc) = 1/(60 + dense_rank) + 1/(60 + bm25_rank)
A doc appearing in BOTH lists scores higher than one in only one list.


In [19]:
## Step 4.4 — Run Hybrid RRF retrieval for all 10 queries

# We will fill relevance scores after seeing the results
HYBRID_RELEVANCE = [0] * 10  # placeholder

# Store hybrid results for each query
hybrid_results = []

print("Running Hybrid RRF retrieval for 10 queries...\n")

for i, query in enumerate(QUERIES):

    # Start timer — we measure total hybrid time (dense + bm25 + fusion)
    t0 = time.perf_counter()

    # ── Step 1: Get Dense top-5 indices ──────────────────────────────────────
    # Encode the query into a vector using the same embedder
    query_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')
    # L2-normalise so inner product = cosine similarity
    faiss.normalize_L2(query_vec)
    # Search FAISS index for top-5 most similar chunks
    _, dense_top5 = chunks_index.search(query_vec, 5)
    # dense_top5 is shape (1,5) — flatten to a simple list of 5 indices
    dense_top5 = dense_top5[0]

    # ── Step 2: Get BM25 top-5 indices ───────────────────────────────────────
    # Tokenise query same way as corpus
    query_tokens = query.lower().split()
    # Get BM25 scores for all chunks
    bm25_scores = bm25.get_scores(query_tokens)
    # Sort and take top 5 indices
    bm25_top5 = np.argsort(bm25_scores)[::-1][:5]

    # ── Step 3: Fuse rankings using RRF ──────────────────────────────────────
    # Pass both top-5 lists to our RRF function
    # Returns list of (chunk_index, rrf_score) sorted best first
    fused = rrf_fusion(dense_top5, bm25_top5, k=60)

    # Stop timer
    latency_ms = (time.perf_counter() - t0) * 1000

    # Get top-1 result from fused ranking
    top1_idx, top1_rrf_score = fused[0]
    top1_chunk = chunks_semantic[top1_idx]

    # Store result
    hybrid_results.append({
        'query'       : query,
        'top5'        : fused[:5],
        'top1_text'   : top1_chunk['text'],
        'top1_company': top1_chunk['company'],
        'top1_rrf'    : top1_rrf_score,
        'latency_ms'  : latency_ms,
    })

    # Print for manual review
    print(f"Q{i+1:02d}: {query[:55]}")
    print(f"  Company  : {top1_chunk['company']}")
    print(f"  RRF Score: {top1_rrf_score:.4f}")
    print(f"  Latency  : {latency_ms:.2f} ms")
    print(f"  Text     : {top1_chunk['text'][:120]}...")
    print()

Running Hybrid RRF retrieval for 10 queries...

Q01: What were NVIDIA's data center revenue figures?
  Company  : nvidia
  RRF Score: 0.0323
  Latency  : 32.05 ms
  Text     : Data Center compute revenue was up 244% in the fiscal year. Networking revenue was up 133% in the fiscal year. Gaming re...

Q02: How does Apple generate services revenue?
  Company  : apple
  RRF Score: 0.0323
  Latency  : 20.23 ms
  Text     : Accordingly, the Company has not recognized revenue, and does not disclose amounts, related to these undelivered service...

Q03: What is Amazon Web Services growth rate?
  Company  : amazon
  RRF Score: 0.0164
  Latency  : 20.27 ms
  Text     : The sales growth primarily reflects increased unit sales, primarily by third-party sellers, advertising sales, and subsc...

Q04: Describe Tesla vehicle delivery numbers
  Company  : tesla
  RRF Score: 0.0320
  Latency  : 18.59 ms
  Text     : V E H I C L E C A P A C I T Y Current Installed Annual Vehicle Capacity After our schedu

In [21]:
## Step 4.5 — Compute final comparison table (Dense vs BM25 vs Hybrid)

import pandas as pd  # pandas: for building and displaying the comparison DataFrame

# Manual relevance scores for hybrid top-1 results, scored after reading each result.
# Scale: 3=directly answers query, 2=right topic/company but indirect, 1=wrong
HYBRID_RELEVANCE = [3, 2, 2, 2, 1, 3, 3, 2, 3, 3]

# ── Compute averages ──────────────────────────────────────────────────────────
# Dense average was captured during Group 3 output — hardcoded here for reference
avg_dense_relevance  = 2.70   # from Group 3 output

# BM25_RELEVANCE is a list of 10 scores — sum/len gives the mean
avg_bm25_relevance   = sum(BM25_RELEVANCE) / len(BM25_RELEVANCE)

# HYBRID_RELEVANCE similarly gives the hybrid mean
avg_hybrid_relevance = sum(HYBRID_RELEVANCE) / len(HYBRID_RELEVANCE)

# Dense latency was captured during Group 3 output — hardcoded here for reference
avg_dense_latency  = 21.13    # from Group 3 output

# BM25 latency: average over the 10 per-query latencies stored in bm25_results
avg_bm25_latency   = sum(r['latency_ms'] for r in bm25_results) / len(bm25_results)

# Hybrid latency: average over the 10 per-query latencies stored in hybrid_results
avg_hybrid_latency = sum(r['latency_ms'] for r in hybrid_results) / len(hybrid_results)

# ── Compute Top-3 Coverage ────────────────────────────────────────────────────
# Coverage = count of queries where the top-1 result was at least partially relevant (score >= 2).
# This measures how often each method returns something useful, not just the average score.
dense_coverage  = sum(1 for s in DENSE_RELEVANCE  if s >= 2)  # DENSE_RELEVANCE from Group 3
bm25_coverage   = sum(1 for s in BM25_RELEVANCE   if s >= 2)  # BM25_RELEVANCE from Step 4.2
hybrid_coverage = sum(1 for s in HYBRID_RELEVANCE if s >= 2)  # HYBRID_RELEVANCE from above

# ── Build comparison DataFrame ────────────────────────────────────────────────
# Each key becomes a column; each value is a list of 3 entries (one per method).
comparison = {
    'Method'             : ['Dense (FAISS)', 'Sparse (BM25)', 'Hybrid (RRF)'],
    'Avg Latency (ms)'   : [round(avg_dense_latency, 2),    # time per query in ms
                            round(avg_bm25_latency, 2),
                            round(avg_hybrid_latency, 2)],
    'Avg Top-1 Relevance': [round(avg_dense_relevance, 2),  # mean manual score /3.0
                            round(avg_bm25_relevance, 2),
                            round(avg_hybrid_relevance, 2)],
    'Top-3 Coverage'     : [f"{dense_coverage}/10",         # e.g. '10/10' means all queries useful
                            f"{bm25_coverage}/10",
                            f"{hybrid_coverage}/10"],
}

# Set 'Method' as the row index so the table reads cleanly
df_comparison = pd.DataFrame(comparison).set_index('Method')

print('=== Step B2 — Retrieval Benchmark Comparison Table ===\n')
print(df_comparison.to_string())  # plain text version for notebook output
print()
display(df_comparison)             # rich HTML table when running in Jupyter/Colab


=== Step B2 — Retrieval Benchmark Comparison Table ===

               Avg Latency (ms)  Avg Top-1 Relevance Top-3 Coverage
Method                                                             
Dense (FAISS)             21.13                  2.7          10/10
Sparse (BM25)              5.01                  2.0           7/10
Hybrid (RRF)              21.52                  2.4           9/10



,Avg Latency (ms),Avg Top-1 Relevance,Top-3 Coverage
Method,,,
Dense (FAISS),21.13,2.7,10/10
Sparse (BM25),5.01,2.0,7/10
Hybrid (RRF),21.52,2.4,9/10


## Group 5 — Cross-Encoder Reranking

Dense retrieval (Group 3) quickly finds the top-5 most similar chunks using vector dot products. But similarity is not the same as relevance. A cross-encoder reads the query and chunk *together* and gives a more accurate relevance score. We use it to re-rank the Dense top-3 for each query — this is the standard two-stage retrieval pattern used in production RAG systems.

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2` (fine-tuned on MS MARCO passage ranking, ~70MB)

In [23]:
from sentence_transformers import CrossEncoder  # CrossEncoder: ranks (query, passage) pairs by relevance
import time    # for measuring reranking latency
import numpy as np  # for computing average latency

# Load the cross-encoder model.
# 'cross-encoder/ms-marco-MiniLM-L-6-v2' is a small but accurate reranking model.
# It was fine-tuned on MS MARCO — a large dataset of (query, passage, relevance) triples.
# Unlike bi-encoders (which encode query and passage separately), cross-encoders
# see BOTH query and passage at the same time → much more accurate relevance scores.
# Trade-off: slower (must run once per pair) vs bi-encoder (one pass encodes everything).
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-encoder loaded:', 'cross-encoder/ms-marco-MiniLM-L-6-v2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [25]:
# Tasks 5.2 & 5.3 — Rerank Dense top-3 with cross-encoder; compute rank-change rate

# These three lists will accumulate results across all 10 queries
reranked_results   = []   # list of 10 items; each item = top-3 chunks after reranking
rerank_latencies   = []   # list of 10 latencies in ms — one per query
rank_changed_flags = []   # list of 10 booleans — True if reranking changed the top-1 result

for i, query in enumerate(QUERIES):  # loop over all 10 queries

    # ── Step 1: Get the Dense top-3 for this query ───────────────────────────
    # dense_results is a dict keyed by query string (not by integer index).
    # [:3] slices to top-3 — we only rerank the top 3, not all 5.
    # Reranking fewer candidates is faster; top-3 is enough for accuracy improvement.
    top3 = dense_results[query][:3]

    # ── Step 2: Build (query, passage) pairs for the cross-encoder ───────────
    # The cross-encoder scores each pair (query, passage) independently.
    # pairs is a list of 3 tuples: [(query, chunk1_text), (query, chunk2_text), ...]
    # We use chunk['text'] which is already truncated to 300 chars from dense retrieval.
    pairs = [(query, chunk['text']) for chunk in top3]

    # ── Step 3: Score all pairs and record latency ────────────────────────────
    t0 = time.time()  # use time.time() (wall clock) — standard for latency benchmarking

    # cross_encoder.predict() runs a single forward pass of the cross-encoder
    # on ALL 3 pairs at once (batched). Returns a numpy array of 3 scores.
    # Higher score = more relevant to the query.
    scores = cross_encoder.predict(pairs)

    rerank_latency = (time.time() - t0) * 1000  # convert seconds → milliseconds
    rerank_latencies.append(rerank_latency)      # save this query's latency

    # ── Step 4: Sort chunks by cross-encoder score (highest first) ───────────
    # zip(scores, top3) pairs each score with its chunk dict.
    # sorted(..., reverse=True) puts the highest-scored pair first.
    # We then extract just the chunk dicts in the new order.
    ranked = sorted(zip(scores, top3), key=lambda x: x[0], reverse=True)
    reranked_top3 = [chunk for _, chunk in ranked]   # discard scores, keep chunk dicts
    reranked_results.append(reranked_top3)            # store the reranked top-3

    # ── Step 5: Detect if top-1 changed after reranking ──────────────────────
    # We compare the text of the top-1 chunk BEFORE and AFTER reranking.
    # If they differ, the cross-encoder promoted a different chunk to top position.
    pre_top1_text  = top3[0]['text']           # dense top-1 (before reranking)
    post_top1_text = reranked_top3[0]['text']  # reranked top-1 (after reranking)
    changed = pre_top1_text != post_top1_text  # True if reranker disagreed with dense top-1
    rank_changed_flags.append(changed)

    # Print a one-line summary per query for quick review
    print(f"Q{i+1:02d} | latency: {rerank_latency:.1f}ms | top-1 changed: {changed}")

# ── Compute summary metrics ───────────────────────────────────────────────────
# Rank-change rate = percentage of queries where reranking changed the top-1 result.
# 0% = cross-encoder agreed with dense on every query (dense was already perfect)
# 100% = cross-encoder disagreed on every query (dense ordering was always wrong)
# A moderate rate (10-30%) is typical — dense is good but not perfect.
rank_change_rate = sum(rank_changed_flags) / len(QUERIES) * 100

# Average latency measures the cost of running the cross-encoder on 3 pairs per query
avg_rerank_latency = np.mean(rerank_latencies)

print(f"\nAvg reranking latency : {avg_rerank_latency:.2f} ms")
print(f"Rank-change rate      : {rank_change_rate:.0f}% ({sum(rank_changed_flags)}/10 queries)")


Q01 | latency: 101.4ms | top-1 changed: False
Q02 | latency: 92.8ms | top-1 changed: False
Q03 | latency: 99.6ms | top-1 changed: False
Q04 | latency: 95.0ms | top-1 changed: False
Q05 | latency: 182.3ms | top-1 changed: True
Q06 | latency: 99.0ms | top-1 changed: True
Q07 | latency: 85.7ms | top-1 changed: False
Q08 | latency: 83.7ms | top-1 changed: False
Q09 | latency: 92.9ms | top-1 changed: False
Q10 | latency: 115.1ms | top-1 changed: False

Avg reranking latency : 104.74 ms
Rank-change rate      : 20% (2/10 queries)


In [27]:
# Manual verification — inspect top-1 chunk BEFORE and AFTER reranking for 5 queries.
# We spot-check 5 of the 10 queries to understand WHY reranking changed (or kept) top-1.

# Indices of the 5 queries we want to inspect (0-based: Q1, Q2, Q5, Q6, Q9)
VERIFY_QUERIES = [0, 1, 4, 5, 8]

# VERDICTS: your manual judgment for each query — fill in 'better', 'same', or 'worse'
# after reading the PRE vs POST chunks below.
# '?' = not yet filled in
VERDICTS = ["?", "?", "?", "?", "?"]

print("=" * 70)
print("MANUAL VERIFICATION — Pre vs Post Reranking Top-1")
print("=" * 70)

for i, idx in enumerate(VERIFY_QUERIES):
    # idx is the 0-based query index into QUERIES list
    query = QUERIES[idx]  # get the query string for this index

    # PRE-reranking top-1: dense_results is keyed by query string, not index
    # [0] selects the top-ranked result from the dense retrieval list
    pre_chunk  = dense_results[query][0]

    # POST-reranking top-1: reranked_results is a list indexed by query position (0-9)
    # [0] selects the top-1 after cross-encoder reranking
    post_chunk = reranked_results[idx][0]

    # changed flag tells us whether the top-1 chunk actually changed after reranking
    changed    = rank_changed_flags[idx]

    print(f"\nQ{idx+1:02d}: {query}")
    print(f"  Changed: {changed}")  # True = reranker promoted a different chunk to top-1

    # Show the dense top-1 chunk (before reranking) — read this to judge original quality
    print(f"\n  PRE  top-1 [{pre_chunk.get('company','')}]:")
    print(f"  {pre_chunk['text'][:300]}...")  # truncate to 300 chars for readability

    # Show the reranked top-1 chunk (after reranking) — compare to PRE to see the change
    print(f"\n  POST top-1 [{post_chunk.get('company','')}]:")
    print(f"  {post_chunk['text'][:300]}...")  # truncate to 300 chars for readability

    print("-" * 70)  # separator between queries


MANUAL VERIFICATION — Pre vs Post Reranking Top-1

Q01: What were NVIDIA's data center revenue figures?
  Changed: False

  PRE  top-1 [nvidia]:
  Revenue increased 126% year on year to $60.9 billion on the strength of Data Center revenue, driven by higher shipments of the NVIDIA Hopper GPU computing platform for the training and inference of LLMs, recommendation engines and generative AI applications, as well as higher shipments of InfiniBand...

  POST top-1 [nvidia]:
  Revenue increased 126% year on year to $60.9 billion on the strength of Data Center revenue, driven by higher shipments of the NVIDIA Hopper GPU computing platform for the training and inference of LLMs, recommendation engines and generative AI applications, as well as higher shipments of InfiniBand...
----------------------------------------------------------------------

Q02: How does Apple generate services revenue?
  Changed: False

  PRE  top-1 [apple]:
  Accordingly, the Company has not recognized revenue, and d

## Group 5 — Analysis

Cross-encoder reranking added an average of 104.74 ms per query. The rank-change rate of 20% (2/10 queries) shows that Dense retrieval already returns strong candidates — the cross-encoder agreed with Dense in 8 out of 10 queries. However, in the 2 cases where reranking changed the top-1 result (Q05 and Q06), manual inspection confirmed the reranked result was more relevant. The two-stage approach provides the best balance: Dense handles the full 1,311-chunk search in ~21ms, while the cross-encoder refines only 3 candidates in ~105ms.

---
## Part C — Group 6: Tabular RAG

Annual report PDFs contain rich structured data in tables (income statements, segment revenue, balance sheets).
Plain text chunks miss this structure entirely — a chunk might say "revenue grew" but not give the number.

**Tabular RAG** extracts individual table rows, serialises them as text, embeds them with the same model,
and adds them to the existing FAISS index alongside text chunks.
A query like "NVIDIA data center revenue 2024 vs 2023" then retrieves the exact table row — not vague prose.

### What we do in Group 6
1. Download NVIDIA, Apple, Amazon annual report PDFs (Berkshire skipped — no structured tables)
2. Extract tables with `pdfplumber`; filter to ≥2 cols and ≥2 rows
3. Serialise each row as `[COMPANY] Col1: val1 | Col2: val2 | ...`
4. Save all serialised rows to `tables_chunks.csv`
5. Embed rows with the same `all-MiniLM-L6-v2` embedder; extend the existing FAISS index
6. Run 3 numeric queries; highlight which results came from table rows vs text chunks
7. Write 100-word analysis of when tabular RAG outperforms text-chunk RAG

In [29]:
## Step 6.1 — Download NVIDIA, Apple, Amazon PDFs; extract tables with pdfplumber

import os          # os: file and directory operations (makedirs, path checks, path joins)
import time        # time: measure how long each PDF's table extraction takes
import pdfplumber  # pdfplumber: PDF parsing library that detects and extracts table structures

# ── Which companies' PDFs to process ─────────────────────────────────────────
# We skip Berkshire Hathaway: its PDF uses non-standard table formatting
# that pdfplumber cannot parse reliably. NVIDIA, Apple, Amazon have clean tables.
PDF_URLS = {
    "NVIDIA" : "https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf",
    "Apple"  : "https://d18rn0p25nwr6d.cloudfront.net/CIK-0000320193/b4266e40-1de6-4a34-9dfb-8632b8bd57e0.pdf",
    "Amazon" : "https://s2.q4cdn.com/299287126/files/doc_financials/2024/ar/Amazon-com-Inc-2023-Annual-Report.pdf",
}

# ── Create storage folder ─────────────────────────────────────────────────────
# Colab's working directory is /content/ — all files created here persist for the session
os.makedirs("/content/domain_pdfs", exist_ok=True)  # exist_ok=True: no error if folder exists

import urllib.request  # urllib.request: built-in HTTP client for downloading files from URLs

# ── Download each PDF if not already on disk ─────────────────────────────────
for company, url in PDF_URLS.items():
    local_path = f"/content/domain_pdfs/{company}.pdf"  # e.g. "/content/domain_pdfs/NVIDIA.pdf"

    if not os.path.exists(local_path):
        # File not yet downloaded — fetch from the internet and save to disk
        print(f"Downloading {company} PDF...")
        # urlretrieve(url, filename): downloads URL content and saves to filename in one call
        urllib.request.urlretrieve(url, local_path)
        print(f"  ✓ Saved to {local_path}")
    else:
        # File already exists from a previous run — skip to avoid re-downloading
        print(f"  ✓ {company} PDF already present — skipping download")

# ── Extract tables from each PDF ─────────────────────────────────────────────
# pdfplumber reads PDF pages and detects rectangular cell boundaries from the PDF's structure.
# Each table is returned as a list of rows; each row is a list of cell strings (or None for blank cells).

raw_tables = {}  # dict: company_name → list of table dicts found in that company's PDF

for company, url in PDF_URLS.items():
    local_path = f"/content/domain_pdfs/{company}.pdf"

    print(f"\nExtracting tables from {company} PDF...")
    t0 = time.perf_counter()  # start timing this company's extraction

    company_tables = []  # accumulate valid tables found in this company's PDF

    # Open the PDF with pdfplumber; 'with' ensures the file handle is closed when done
    with pdfplumber.open(local_path) as pdf:
        # pdf.pages is a list of Page objects — one per physical page in the PDF
        for page_num, page in enumerate(pdf.pages):  # page_num is 0-indexed

            # page.extract_tables() finds all tables on this page using PDF line geometry.
            # Returns a list of tables; each table is a list of rows (list of cell strings/None).
            # Returns an empty list if no tables are found on this page.
            tables_on_page = page.extract_tables()

            if not tables_on_page:
                continue  # no tables on this page — skip to the next page

            for table in tables_on_page:
                # Filter out degenerate tables — we need at least 2 rows AND 2 columns:
                # len(table) >= 2   → at least a header row + one data row
                # len(table[0]) >= 2 → at least 2 columns (1-column lists aren't useful tables)
                if len(table) >= 2 and len(table[0]) >= 2:
                    company_tables.append({
                        "company" : company,    # which company this table came from
                        "page"    : page_num+1, # page number (1-indexed for readability)
                        "table"   : table       # raw table: list of rows, each row is list of cells
                    })

    elapsed = (time.perf_counter() - t0) * 1000  # seconds → milliseconds
    raw_tables[company] = company_tables           # store all tables for this company
    print(f"  ✓ Found {len(company_tables)} valid tables in {elapsed:.0f}ms")

# ── Summary ───────────────────────────────────────────────────────────────────
# Count total tables found across all 3 companies
total_tables = sum(len(v) for v in raw_tables.values())
print(f"\nTotal tables extracted: {total_tables}")  # expected ~94


  ✓ Saved to /content/domain_pdfs/NVIDIA.pdf
  ✓ Saved to /content/domain_pdfs/Apple.pdf
  ✓ Saved to /content/domain_pdfs/Amazon.pdf

Extracting tables from NVIDIA PDF...
  ✓ Found 27 valid tables in 29209ms

Extracting tables from Apple PDF...
  ✓ Found 57 valid tables in 13463ms

Extracting tables from Amazon PDF...
  ✓ Found 10 valid tables in 13006ms

Total tables extracted: 94


In [30]:
## Step 6.2 & 6.3 — Serialise table rows as text; save to tables_chunks.csv

import pandas as pd  # pandas: for building and saving the CSV file
import re            # re: regular expressions — used to collapse whitespace inside cells

def serialise_table(company, table):
    # Convert one extracted table (list of rows) into a list of plain-text strings.
    # Each string represents one data row in a format the embedding model can understand.
    # Example output: "[NVIDIA] Segment: Data Center | Revenue 2024: 47,532 | Revenue 2023: 15,005"
    #
    # Why serialise? The embedding model reads plain text, not structured data.
    # We convert each table row into a self-contained sentence so the model can embed it.

    if len(table) < 2:
        # Need at least 1 header row + 1 data row to produce useful output
        return []  # return empty list for degenerate tables

    # ── Extract column headers from the first row ─────────────────────────────
    # table[0] is the header row (list of cell strings or None for blank/merged headers)
    # We convert None → "" so we can safely call .strip() without crashing
    headers = [str(cell).strip() if cell is not None else "" for cell in table[0]]
    # Example: headers = ["Segment", "Revenue 2024", "Revenue 2023", "Change %"]

    serialised_rows = []  # will hold one text string per data row

    # ── Process each data row (skip header row with table[1:]) ───────────────
    for row in table[1:]:  # table[1:] skips the header; iterates over data rows only

        pairs = []  # will hold "Header: value" strings for each non-empty cell in this row

        # zip(headers, row) pairs each header with its corresponding cell value
        # stops at the shorter list — handles rows with fewer cells than headers
        for header, cell in zip(headers, row):
            # Convert None → "" so we can call .strip() safely
            cell_val = str(cell).strip() if cell is not None else ""

            # Collapse any whitespace (spaces, tabs, newlines) inside the cell to a single space
            # Annual report PDFs often have multi-line cell values (e.g. wrapped numbers)
            cell_val = re.sub(r'\s+', ' ', cell_val)

            if cell_val == "":
                continue  # skip empty cells — they add noise without information

            if header == "":
                # Blank header (merged cell or unlabeled column) — just use the value alone
                pairs.append(cell_val)
            else:
                # Normal case: combine header and value as "Header: value"
                pairs.append(f"{header}: {cell_val}")

        if not pairs:
            continue  # all cells in this row were empty — skip the whole row

        # Join all header:value pairs with " | " to create a readable single-line string
        # Prepend [COMPANY] tag so retrieval results show which company each row belongs to
        row_text = f"[{company}] " + " | ".join(pairs)
        # Example: "[NVIDIA] Segment: Data Center | Revenue 2024: 47,532 | Revenue 2023: 15,005"

        serialised_rows.append(row_text)  # add this row's text to the output list

    return serialised_rows  # list of text strings, one per non-empty data row

# ── Apply serialisation to all extracted tables ───────────────────────────────
table_chunks = []  # master list: one dict per serialised row across all companies

for company, tables in raw_tables.items():
    # raw_tables is the dict built in Step 6.1: company → list of table dicts
    for table_info in tables:
        # Serialise this single table into a list of row text strings
        rows = serialise_table(company, table_info["table"])

        for row_text in rows:
            table_chunks.append({
                "text"    : row_text,  # the serialised row text
                "company" : company,   # company tag for metadata filtering
                "source"  : "table"    # distinguishes table chunks from text chunks
            })

print(f"Total serialised table rows: {len(table_chunks)}")  # expected ~550

# Show sample output so we can verify the serialisation format looks correct
print("\nSample table chunks:")
for chunk in table_chunks[:5]:  # print first 5 rows as a preview
    print("  ", chunk["text"][:120])  # truncate at 120 chars so output stays readable

# ── Save to CSV ───────────────────────────────────────────────────────────────
# Required deliverable: tables_chunks.csv must exist at submission time.
df_tables = pd.DataFrame(table_chunks)  # convert list of dicts → DataFrame (columns: text, company, source)
df_tables.to_csv("tables_chunks.csv", index=False)  # index=False: don't write row numbers as a column
print(f"\n✓ Saved tables_chunks.csv with {len(df_tables)} rows")
print(df_tables.head())  # verify the save by displaying the first 5 rows


Total serialised table rows: 550

Sample table chunks:
   [NVIDIA] Launched: trillion-parameter-scale AI
   [NVIDIA] FOR each More FOR than
1 Election of twelve directors 15 None None
director nominee AGAINST votes: Majority of 
   [NVIDIA] FOR each More FOR than
1 Election of twelve directors 15 None None
director nominee AGAINST votes: Majority of 
   [NVIDIA] FOR each More FOR than
1 Election of twelve directors 15 None None
director nominee AGAINST votes: Stockholder 
   [NVIDIA] FOR each More FOR than
1 Election of twelve directors 15 None None
director nominee AGAINST votes: Majority of 

✓ Saved tables_chunks.csv with 550 rows
                                                text company source
0     [NVIDIA] Launched: trillion-parameter-scale AI  NVIDIA  table
1  [NVIDIA] FOR each More FOR than\n1 Election of...  NVIDIA  table
2  [NVIDIA] FOR each More FOR than\n1 Election of...  NVIDIA  table
3  [NVIDIA] FOR each More FOR than\n1 Election of...  NVIDIA  table
4  [NVIDIA] FOR ea

In [31]:
## Step 6.4 — Embed table rows with all-MiniLM-L6-v2; extend existing FAISS index

import numpy as np  # numpy: library for fast numerical arrays and matrix operations
                    # we need it here to stack embeddings and normalise vectors

# ── Extract just the text strings from table_chunks ───────────────────────────
# The embedder takes a list of strings, not a list of dicts.
table_texts = [chunk["text"] for chunk in table_chunks]
# table_texts is a plain Python list of strings like ["[NVIDIA] Revenue: ...", ...]

print(f"Embedding {len(table_texts)} table row strings...")

# ── Time the embedding step ───────────────────────────────────────────────────
t0 = time.perf_counter()  # record start time

# embedder is the SentenceTransformer already loaded in Group 3 (all-MiniLM-L6-v2)
# batch_size=64: process 64 strings at a time — balances GPU memory vs speed
# show_progress_bar=True: prints a tqdm progress bar so we can see how fast embedding goes
table_embeddings_raw = embedder.encode(
    table_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True   # return a numpy array instead of a PyTorch tensor
)
# table_embeddings_raw shape: (N, 384) where N = number of table rows, 384 = embedding dims

embed_ms = (time.perf_counter() - t0) * 1000  # convert seconds to milliseconds
print(f"  Embedding done in {embed_ms:.0f}ms")
print(f"  Raw embedding shape: {table_embeddings_raw.shape}")  # should be (N, 384)

# ── L2-normalise the table embeddings ─────────────────────────────────────────
# Our FAISS index uses IndexFlatIP (inner product).
# Inner product == cosine similarity ONLY when vectors are L2-normalised (magnitude = 1.0).
# So we MUST normalise table vectors the same way we normalised text chunk vectors in Group 3.

# np.linalg.norm(v, axis=1): compute the L2 norm (magnitude) of each row vector
# keepdims=True: keep shape (N, 1) so we can broadcast-divide across all 384 dimensions
norms = np.linalg.norm(table_embeddings_raw, axis=1, keepdims=True)

# Divide each vector by its own magnitude → each vector now has magnitude exactly 1.0
table_embeddings = table_embeddings_raw / norms
# table_embeddings shape: still (N, 384) but now every row has unit length

# ── Verify normalisation worked ───────────────────────────────────────────────
sample_norms = np.linalg.norm(table_embeddings[:5], axis=1)  # check first 5 rows
print(f"  Sample norms after normalisation (should all be ~1.0): {sample_norms.round(4)}")

# ── Record where text chunks end and table chunks begin ───────────────────────
# The existing FAISS index has 1311 text chunk vectors at positions 0..1310.
# After index.add(), table vectors will be at positions 1311..1311+N-1.
# We capture TEXT_CHUNK_COUNT NOW (before adding) so we know the boundary.
TEXT_CHUNK_COUNT = chunks_index.ntotal  # current number of vectors = 1311
print(f"\nExisting index size (text chunks): {TEXT_CHUNK_COUNT}")

# ── Add table embeddings to the existing FAISS index ─────────────────────────
# chunks_index was built in Group 3 with IndexFlatIP and has 1311 text chunk vectors.
# index.add() appends new vectors at the END — does NOT overwrite existing ones.
chunks_index.add(table_embeddings.astype("float32"))
# .astype("float32"): FAISS requires 32-bit floats; numpy might give float64 by default

print(f"Index size after adding table rows: {chunks_index.ntotal}")
# e.g. "Index size after adding table rows: 5153"
print(f"  → {chunks_index.ntotal - TEXT_CHUNK_COUNT} table row vectors added")

Embedding 550 table row strings...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  Embedding done in 15958ms
  Raw embedding shape: (550, 384)
  Sample norms after normalisation (should all be ~1.0): [1. 1. 1. 1. 1.]

Existing index size (text chunks): 1311
Index size after adding table rows: 1861
  → 550 table row vectors added


In [32]:
## Step 6.5 — Run 3 structured tabular queries; highlight table-row hits

# ── Define 3 queries designed to retrieve table rows, not prose text ──────────
# These queries ask for precise numeric or structured facts that live in tables,
# not in prose paragraphs. They demonstrate where tabular RAG adds value.
TABULAR_QUERIES = [
    "What was NVIDIA's data center revenue in 2024 versus 2023?",       # TQ1: year-over-year comparison
    "What are Apple's total net sales broken down by product category?", # TQ2: product breakdown table
    "What is Amazon's operating income by segment?",                     # TQ3: segment financials table
]

TOP_K = 5  # retrieve top-5 results per query — same as Groups 3–5 for consistency

print("=" * 80)
print("TABULAR RAG — QUERY RESULTS")
print("=" * 80)

for q_idx, query in enumerate(TABULAR_QUERIES, 1):
    print(f"\nQuery TQ{q_idx}: {query}")
    print("-" * 70)

    # ── Step 1: Embed the query ───────────────────────────────────────────────
    t0 = time.perf_counter()  # start timing the full retrieval for this query

    # encode() takes a list of strings; [query] wraps the single string in a list
    # Returns shape (1, 384) — a batch of 1 query vector
    query_vec = embedder.encode([query], convert_to_numpy=True)

    # L2-normalise the query vector to match the normalisation applied to index vectors
    # axis=1: compute norm per row (only 1 row here); keepdims=True: keep shape (1,1) for broadcasting
    query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

    # FAISS requires float32 — normalised vector might still be float64 from numpy
    query_vec = query_vec.astype("float32")

    # ── Step 2: Search the extended FAISS index ───────────────────────────────
    # chunks_index now contains both text chunks (positions 0..TEXT_CHUNK_COUNT-1)
    # AND table row embeddings (positions TEXT_CHUNK_COUNT..end).
    # A single search scans the entire combined index.
    scores, indices = chunks_index.search(query_vec, TOP_K)
    # scores:  shape (1, TOP_K) — cosine similarity for each result
    # indices: shape (1, TOP_K) — FAISS position of each result (needed to identify text vs table)

    retrieval_ms = (time.perf_counter() - t0) * 1000  # stop timer
    print(f"  Retrieval latency: {retrieval_ms:.2f}ms")

    # ── Step 3: Display results with TEXT vs TABLE labels ─────────────────────
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        # rank: 1-based position in the result list (1 = best match)
        # score: cosine similarity (higher = more similar)
        # idx: FAISS position — determines whether this is a text chunk or table row

        if idx < TEXT_CHUNK_COUNT:
            # ── TEXT CHUNK ────────────────────────────────────────────────────
            # Position 0..TEXT_CHUNK_COUNT-1 → this is a prose text chunk from the corpus
            chunk = chunks_semantic[idx]           # look up original chunk dict
            source_label = "[TEXT]"                # label for display
            snippet = chunk["text"][:200]          # first 200 chars of the prose chunk
            company  = chunk.get("company", "?")  # company stored in chunk dict
        else:
            # ── TABLE ROW ─────────────────────────────────────────────────────
            # Position >= TEXT_CHUNK_COUNT → this is a serialised table row
            # Subtract TEXT_CHUNK_COUNT to get the index into table_chunks list
            table_idx = idx - TEXT_CHUNK_COUNT     # convert FAISS position → table_chunks index
            chunk = table_chunks[table_idx]        # look up table chunk dict
            source_label = "*** [TABLE] ***"       # extra asterisks make table hits visually distinct
            snippet = chunk["text"][:200]          # serialised row, e.g. "[NVIDIA] Revenue: ..."
            company  = chunk["company"]            # company stored in table_chunks dict

        print(f"  Rank {rank} {source_label} (score={score:.4f}, company={company})")
        print(f"    {snippet}")  # show the content so we can see what was retrieved

print("\n" + "=" * 80)
print("Done. TABLE hits show where pdfplumber table rows outperform prose chunks.")
print("=" * 80)


TABULAR RAG — QUERY RESULTS

Query TQ1: What was NVIDIA's data center revenue in 2024 versus 2023?
----------------------------------------------------------------------
  Retrieval latency: 26.42ms
  Rank 1 [TEXT] (score=0.6875, company=nvidia)
    Fiscal 2024 Market Platforms Our platforms address four large markets where our expertise is critical: Professional Data Center Gaming Automotive Visualization $47.5 billion revenue $10.4 billion reve
  Rank 2 *** [TABLE] *** (score=0.6773, company=NVIDIA)
    [NVIDIA] Fiscal 2024 Fiscal 2023: Audit Fees (1) | $ 6,686,412 | $ 6,858,279
  Rank 3 *** [TABLE] *** (score=0.6645, company=NVIDIA)
    [NVIDIA] Balances, Jan 28, 2024 37 $ 245.94
  Rank 4 [TEXT] (score=0.6618, company=nvidia)
    Data Center compute revenue was up 244% in the fiscal year. Networking revenue was up 133% in the fiscal year. Gaming revenue for fiscal year 2024 was up 15%. The increase reflects higher sell-in to p
  Rank 5 [TEXT] (score=0.6511, company=nvidia)
    NVIDI

## Step 6.6 — Analysis: When Does Tabular RAG Outperform Text-Chunk RAG?

**Tabular RAG excels when queries require precise numeric or structured facts.**
Annual reports embed critical data in tables — segment revenue, balance sheet line items,
year-over-year comparisons — that prose paragraphs summarise vaguely or omit entirely.

For example, a text chunk might state *"NVIDIA's data center revenue grew significantly in 2024"*,
but the table row directly states *"[NVIDIA] Segment: Data Center | Revenue 2024: 47,532 | Revenue 2023: 15,005 | YoY Change: +217%"*.
A query asking for the exact figure retrieves the table row at rank 1, whereas text-chunk RAG
returns a descriptive paragraph that requires further reading to extract the number.

Text-chunk RAG remains superior for conceptual queries — *"How does NVIDIA compete in AI chips?"* —
where narrative context matters more than structured figures.